## Code Review RAG

Este notebook demonstra como criar um sistema RAG (Retrieval-Augmented Generation) focado em **Revisão de Código**.

O sistema:
1. Clona um repositório GitHub (LangChain).
2. Carrega arquivos de código Python.
3. Divide o código em pedaços (chunks).
4. Cria embeddings e armazena em um banco vetorial (ChromaDB).
5. Usa uma LLM para responder perguntas e revisar o código com base no contexto do repositório.

### 1. Importações
Carregamos todas as bibliotecas necessárias, incluindo LangChain, GitPython e ChromaDB.


In [ ]:
from langchain_core.runnables import RunnableBinding
from git import Repo
from langchain_chroma import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter



### 2. Configuração de Ambiente
Carrega as variáveis de ambiente (como `OPENAI_API_KEY`) do arquivo `.env`.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

### 3. Clonar Repositório
Clona o repositório do **LangChain** para análise. Se a pasta já existir, ele pode dar erro ou usar a existente (depende do estado, aqui estamos clonando para `./test_repo`).

In [34]:
# Define o caminho local onde o repo será salvo e a URL do repo
repo_path = './test_repo'
gh_path = 'https://github.com/langchain-ai/langchain'

# Clona o repositório do GitHub para o diretório local especificiado
repo = Repo.clone_from(gh_path, repo_path)


### 4. Carregar Documentos de Código
Usa o `GenericLoader` para varrer os arquivos Python do repositório clonado. Filtra apenas arquivos `.py` e ignora diretórios irrelevantes.

In [35]:
# Configura o loader para buscar arquivos Python recursivamente
loader = GenericLoader.from_filesystem(
  repo_path + '/libs/langchain/langchain_classic/chat_models', # Caminho base
  glob='**/*.py', # Padrão de busca (todos .py)
  suffixes=['.py'], # Extensões permitidas
  exclude=['.venv', '.env', '__pycache__', 'node_modules', '.git'], # Pastas ignoradas
  parser=LanguageParser(language=Language.PYTHON, parser_threshold=500) # Parser específico para Python
)

# Executa o carregamento dos documentos
docs = loader.load()

f'Documentos carregados: {len(docs)}'

'Documentos carregados: 44'

### 5. Dividir Código (Splitting)
Para processar arquivos grandes, dividimos o código em pedaços menores (chunks) respeitando a sintaxe Python (`RecursiveCharacterTextSplitter.from_language`).

In [36]:
# Configura o splitter de texto ciente da linguagem Python
python_splitter = RecursiveCharacterTextSplitter.from_language(
  language=Language.PYTHON,
  chunk_size=1700, # Tamanho máximo de cada chunk
  chunk_overlap=300 # Sobreposição para manter contexto entre chunks
)

# Divide os documentos carregados em chunks menores
texts = python_splitter.split_documents(docs)

f'Chunks gerados: {len(texts)}'

'Chunks gerados: 69'

### 6. Criação de Embeddings e Banco Vetorial
Transformamos os chunks de texto em vetores numéricos usando `OpenAIEmbeddings` e salvamos no `ChromaDB` para busca eficiente.

In [37]:
# Inicializa o modelo de embeddings da OpenAI
embedding = OpenAIEmbeddings(
  model="text-embedding-ada-002",
  disallowed_special=() # Permite caracteres especiais se necessário
)

In [38]:
# Cria (ou carrega) o banco de dados vetorial Chroma com os documentos processados
db = Chroma.from_documents(texts, embedding, persist_directory="../databases/db_code_review")

### 7. Configurar Retriever
O retriever é responsável por buscar os chunks mais relevantes para a pergunta do usuário. Usamos `mmr` (Maximal Marginal Relevance) para diversificar os resultados.

In [39]:
# Configura o retriever (buscador)
# mmr: Busca por relevância máxima marginal (evita documentos muito parecidos)
# k: 8: Retorna os 8 chunks mais relevantes
retriever = db.as_retriever(
  search_type="mmr",
  search_kwargs={"k": 8}
)

### 8. Configurar LLM e Prompt
Definimos o modelo (`gpt-3.5-turbo`) e o prompt que instrui a IA a agir como um revisor de código sênior.

In [61]:
# Inicializa o modelo de chat (LLM)
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7, max_tokens=1000) 

In [53]:
# Define o template do prompt para o chat
prompt = ChatPromptTemplate.from_messages([
    # Mensagem do sistema com instruções de persona e contexto
    ("system", """
    Você é um desenvolvedor sênior com larga experiência na área de IA 
    e revisor de código experiente.
    
    Analise o código do usuário com base no seguinte contexto recuperado:
    {context}
    
    Forneça informações detalhadas e feedbacks sobre o código.
    """),
    # Entrada do usuário (pergunta sobre o código)
    ("human", "{input}")
])



### 9. Criar Chains
Criamos a **Document Chain** (que processa documentos e gera resposta) e a **Retrieval Chain** (que conecta o retriever à document chain).

In [57]:
# Cria a chain que processa os documentos recuperados e gera a resposta
document_chain = create_stuff_documents_chain(
  llm=llm,
  prompt=prompt,
  output_parser=StrOutputParser()
)

# Cria a chain completa de RAG, conectando o retriever e a document chain
retrieval_chain = create_retrieval_chain(
  retriever, document_chain
) 

### 10. Executar Code Review
Fazemos uma pergunta específica sobre o código indexado.

In [1]:
# Invoca a chain com uma pergunta específica
response = retrieval_chain.invoke({
  "input": "Você pode revisar e sugerir melhorias para o código de RunnableBinding dentro do langchain?",
})

NameError: name 'retrieval_chain' is not defined

In [ ]:
# Exibe apenas a resposta gerada pela LLM
response['answer']

'Claro! Vou revisar e fornecer algumas sugestões de melhorias para o código do `RunnableBinding` dentro do langchain:\n\n1. **Documentação e Comentários:**\n    - Adicione mais comentários explicativos em partes do código que possam não estar tão claras.\n    - Certifique-se de que a documentação das funções, propriedades e classes esteja completa e informativa.\n\n2. **Código Limpo e Organizado:**\n    - Mantenha a consistência na formatação do código, como espaçamento, indentação e quebras de linha.\n    - Agrupe logicamente as operações relacionadas juntas para facilitar a leitura e compreensão do código.\n\n3. **Melhorias de Desempenho:**\n    - Considere a possibilidade de otimizar o código para melhorar o desempenho, especialmente em operações que envolvem'

In [ ]:
RunnableBinding